In [1]:
# STEP 1 — Train 295-class classifier (100 epochs per backbone, NO early stop)
#   • Phase 1: 20 epochs, head only (backbone frozen)
#   • Phase 2: 80 epochs, full fine-tune (all layers unfrozen)
#   • Always saves final-epoch checkpoint + best-val-acc checkpoint
#
# STEP 1b — Spoof-Aware Fine-Tuning (30 epochs, NO early stop)
#   • Loads _cls.pt, fine-tunes with triplet + CE + BCE loss
#   • Triplet margin = 0.5 (stronger separation)
#   • Uses image-level pairs (not subject averages)
#   • Saves _sa.pt
#
# STEP 2 — Extract image-level embeddings (NOT just subject averages)
#   • All images per subject → individual embeddings
#   • Also stores ALL individual RS pairs (not just 295)
#   • Subject mean embedding computed for RR and RC
#   • Individual image-level pairs used for RS → ~5900 pairs → smoother orange dist
#
# STEP 3 — Pairwise dissimilarity (fixed d' formula)
#   • Ver A Real-Real: 43,365 pairs (subject-level means)
#   • Ver B Real-Spoof SAME: image-level (~20×295 ≈ 5900 pairs) → smooth orange
#   • Ver C Real-Spoof CROSS: subject-level means
#   • d' = (RR_mean - RS_mean) / pooled_std → POSITIVE when correct


In [2]:
# ── Cell 1: Imports ───────────────────────────────────────────────
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
from pathlib import Path
from PIL import Image
from itertools import combinations
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


torch  : 2.9.1+cu128
CUDA   : True
GPU    : NVIDIA RTX A5000
VRAM   : 25.4 GB


In [3]:
# ── Cell 2: Config & Paths ────────────────────────────────────────
BASE       = '/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean'
REAL_DIR   = os.path.join(BASE, 'real')
ATTACK_DIR = os.path.join(BASE, 'attack')
SAVE_DIR   = os.path.join(BASE, 'outputs_295class')
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 32
WD         = 1e-4
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BACKBONES  = ['ResNet18', 'ResNet50', 'EfficientNetB0', 'InceptionV3']
EXTS       = ['*.jpg','*.jpeg','*.png','*.bmp',
               '*.JPG','*.JPEG','*.PNG','*.BMP']
COLORS     = {
    'ResNet18':       '#4C72B0',
    'ResNet50':       '#DD8452',
    'EfficientNetB0': '#55A868',
    'InceptionV3':    '#C44E52',
}

print(f'Device  : {DEVICE}')
print(f'Real    : {os.path.abspath(REAL_DIR)}')
print(f'Attack  : {os.path.abspath(ATTACK_DIR)}')
print(f'Outputs : {os.path.abspath(SAVE_DIR)}')


Device  : cuda
Real    : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/real
Attack  : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/attack
Outputs : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/outputs_295class


In [4]:
# ── Cell 3: Discover Subjects & Build Class Map ───────────────────
assert os.path.isdir(REAL_DIR),   f'Not found: {REAL_DIR}'
assert os.path.isdir(ATTACK_DIR), f'Not found: {ATTACK_DIR}'

real_subjects   = sorted([s for s in os.listdir(REAL_DIR)
                           if os.path.isdir(os.path.join(REAL_DIR, s))])
attack_subjects = sorted([s for s in os.listdir(ATTACK_DIR)
                           if os.path.isdir(os.path.join(ATTACK_DIR, s))])

SUBJECT_TO_IDX = {s: i for i, s in enumerate(real_subjects)}
NUM_CLASSES    = len(real_subjects)
matched_subjects = sorted(set(real_subjects) & set(attack_subjects))

print(f'Real subjects    : {len(real_subjects)}')
print(f'Attack subjects  : {len(attack_subjects)}')
print(f'Matched subjects : {len(matched_subjects)}')
print(f'NUM_CLASSES      : {NUM_CLASSES}')
print(f'Total pairs RR   : {NUM_CLASSES*(NUM_CLASSES-1)//2:,}')
print(f'Total pairs RS (image-level estimate): ~{len(matched_subjects)*20:,}')


Real subjects    : 295
Attack subjects  : 295
Matched subjects : 295
NUM_CLASSES      : 295
Total pairs RR   : 43,365
Total pairs RS (image-level estimate): ~5,900


In [5]:
# ── Cell 4: Datasets ──────────────────────────────────────────────

class ClassificationDataset(Dataset):
    """Multi-class dataset — real images only, returns (image, class_idx)."""
    def __init__(self, samples, transform, img_size):
        self.samples   = samples
        self.transform = transform
        self.resize    = transforms.Resize((img_size, img_size))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.resize(Image.open(path).convert('RGB'))
        return self.transform(img), torch.tensor(label, dtype=torch.long)


class SubjectImageDataset(Dataset):
    """Returns all images for embedding extraction.
    Returns (image_tensor, subject_name, split_tag)."""
    def __init__(self, samples, transform, img_size):
        self.samples   = samples
        self.transform = transform
        self.resize    = transforms.Resize((img_size, img_size))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, subj, split = self.samples[idx]
        img = self.resize(Image.open(path).convert('RGB'))
        return self.transform(img), subj, split


def build_cls_samples(real_dir, subject_to_idx):
    samples = []
    for subj, idx in subject_to_idx.items():
        sp = Path(real_dir) / subj
        if not sp.is_dir(): continue
        for ext in EXTS:
            for p in sp.glob(ext):
                samples.append((str(p), idx))
    return samples


def build_emb_samples(real_dir, attack_dir, subjects):
    samples = []
    for subj in subjects:
        rp = Path(real_dir)   / subj
        ap = Path(attack_dir) / subj
        if rp.is_dir():
            for ext in EXTS:
                for p in rp.glob(ext):
                    samples.append((str(p), subj, 'real'))
        if ap.is_dir():
            for ext in EXTS:
                for p in ap.glob(ext):
                    samples.append((str(p), subj, 'attack'))
    return samples


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def get_transforms(backbone):
    """InceptionV3 needs 299×299; all others use 224×224."""
    sz = 299 if backbone == 'InceptionV3' else 224
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, eval_tf, sz


# Build classification samples, 80/20 image-level split
all_cls = build_cls_samples(REAL_DIR, SUBJECT_TO_IDX)
train_cls, val_cls = train_test_split(
    all_cls, test_size=0.20, random_state=SEED,
    stratify=[s[1] for s in all_cls]
)

# Build embedding samples for ALL subjects (real + attack)
all_emb_samples = build_emb_samples(REAL_DIR, ATTACK_DIR, real_subjects)

print(f'Classification samples : {len(all_cls)}')
print(f'  Train / Val          : {len(train_cls)} / {len(val_cls)}')
print(f'Embedding samples      : {len(all_emb_samples)}')
print('Dataset helpers ready.')


Classification samples : 5900
  Train / Val          : 4720 / 1180
Embedding samples      : 11800
Dataset helpers ready.


In [6]:
# ── Cell 5: Backbone Builders ─────────────────────────────────────
# FIX: EfficientNetB0 build_extractor now uses a proper projection head
# that matches the embedding space the classifier was trained in.

def build_classifier(backbone, n_cls):
    if backbone == 'ResNet18':
        m    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.4),     nn.Linear(256, n_cls))
        return m

    if backbone == 'ResNet50':
        m    = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),      nn.Linear(512, n_cls))
        return m

    if backbone == 'EfficientNetB0':
        base = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        head = nn.Sequential(
            nn.Linear(1280, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),      nn.Linear(512, n_cls))
        class _EffNet(nn.Module):
            def __init__(self):
                super().__init__()
                self.backbone = base
                self.head     = head
            def forward(self, x): return self.head(self.backbone(x))
        return _EffNet()

    if backbone == 'InceptionV3':
        m              = models.inception_v3(
            weights=models.Inception_V3_Weights.IMAGENET1K_V1,
            aux_logits=True)
        m.fc           = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),      nn.Linear(512, n_cls))
        m.AuxLogits.fc = nn.Linear(768, n_cls)
        return m

    raise ValueError(f'Unknown backbone: {backbone}')


def build_extractor(backbone, ckpt_path):
    """
    Load trained weights, strip classification head.
    FIX for EfficientNetB0: now passes through global_pool so the
    feature space matches what the classifier head was trained on.
    All extractors output raw (un-normalized) vectors;
    L2-normalisation happens separately.
    """
    if backbone == 'ResNet18':
        m    = models.resnet18(weights=None)
        m.fc = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.4),     nn.Linear(256, NUM_CLASSES))
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        ext = nn.Sequential(*list(m.children())[:-1], nn.Flatten())
        return ext.to(DEVICE).eval()

    if backbone == 'ResNet50':
        m    = models.resnet50(weights=None)
        m.fc = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),      nn.Linear(512, NUM_CLASSES))
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        ext = nn.Sequential(*list(m.children())[:-1], nn.Flatten())
        return ext.to(DEVICE).eval()

    if backbone == 'EfficientNetB0':
        # ── FIX: was returning m.backbone which outputs pre-pooled features
        # Now: load full model, then extract backbone + global_pool → 1280-d
        # This is consistent with what the head was trained on.
        base = timm.create_model('efficientnet_b0', pretrained=False, num_classes=0)
        head = nn.Sequential(
            nn.Linear(1280, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),      nn.Linear(512, NUM_CLASSES))
        class _EffNet(nn.Module):
            def __init__(self):
                super().__init__()
                self.backbone = base
                self.head     = head
            def forward(self, x): return self.head(self.backbone(x))
        m = _EffNet()
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        # ── FIXED extractor: backbone already includes global avg pool in timm
        # Returns 1280-d vectors consistent with training embedding space
        class _EffNetExtractor(nn.Module):
            def __init__(self, backbone):
                super().__init__()
                self.bb = backbone
            def forward(self, x):
                feats = self.bb.forward_features(x)   # (B, 1280, 7, 7)
                feats = self.bb.global_pool(feats)     # (B, 1280)
                return feats
        return _EffNetExtractor(m.backbone).to(DEVICE).eval()

    if backbone == 'InceptionV3':
        m              = models.inception_v3(weights=None, aux_logits=True)
        m.fc           = nn.Sequential(
            nn.Linear(2048, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),      nn.Linear(512, NUM_CLASSES))
        m.AuxLogits.fc = nn.Linear(768, NUM_CLASSES)
        m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        m.aux_logits   = False
        m.AuxLogits    = None
        m.fc           = nn.Identity()   # outputs 2048-d
        return m.to(DEVICE).eval()

    raise ValueError(f'Unknown backbone: {backbone}')

print('Backbone builders ready.')


Backbone builders ready.


In [7]:
# ── Cell 6: Train & Eval Helpers ──────────────────────────────────
def train_epoch(model, loader, optimizer, backbone):
    model.train()
    total_loss = 0.0; correct = 0; total = 0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        if backbone == 'InceptionV3':
            out = model(imgs)
            if hasattr(out, 'logits'):
                loss   = F.cross_entropy(out.logits, labels) + \
                         0.4 * F.cross_entropy(out.aux_logits, labels)
                logits = out.logits
            else:
                logits = out; loss = F.cross_entropy(out, labels)
        else:
            logits = model(imgs)
            loss   = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, backbone):
    model.eval()
    total_loss = 0.0; correct = 0; total = 0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if backbone == 'InceptionV3':
            model.aux_logits = False
            logits = model(imgs)
            model.aux_logits = True
        else:
            logits = model(imgs)
        loss        = F.cross_entropy(logits, labels)
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total

print('Train/eval helpers ready.')


Train/eval helpers ready.


In [8]:
# ── Cell 7: STEP 1 — Train All 4 Classifiers (EXACTLY 100 epochs) ─
# Phase 1 : 20 epochs — head only       (backbone frozen)
# Phase 2 : 80 epochs — full fine-tune  (all layers unfrozen)
# FIX 1: Final-epoch checkpoint always saved (guarantees _cls.pt exists)
# FIX 2: SAVE_DIR never overridden after this cell

# Always re-confirm SAVE_DIR is correct before training
SAVE_DIR = os.path.join(BASE, 'outputs_295class')
os.makedirs(SAVE_DIR, exist_ok=True)

all_histories = {}

for bname in BACKBONES:

    print('\n' + '='*62)
    print(f'  TRAINING  {bname}  →  {NUM_CLASSES}-class classification')
    print(f'  Training data : REAL images only')
    print(f'  Phase 1       : 20 epochs  (head only)')
    print(f'  Phase 2       : 80 epochs  (full network)')
    print(f'  Total         : EXACTLY 100 epochs — no early stop')
    print('='*62)

    train_tf, eval_tf, sz = get_transforms(bname)

    tr_ld = DataLoader(
        ClassificationDataset(train_cls, train_tf, sz),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=True)

    vl_ld = DataLoader(
        ClassificationDataset(val_cls, eval_tf, sz),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=True)

    model = build_classifier(bname, NUM_CLASSES).to(DEVICE)
    ckpt  = os.path.join(SAVE_DIR, f'{bname}_cls.pt')
    hist  = []

    n_p = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'\n  Params          : {n_p:.1f}M')
    print(f'  Input size      : {sz}×{sz}')
    print(f'  Random baseline : {100/NUM_CLASSES:.2f}%\n')

    # ── Phase 1: head only, 20 epochs ────────────────────────────
    P1_EP = 20

    if bname in ('ResNet18', 'ResNet50'):
        for n, p in model.named_parameters():
            p.requires_grad = 'fc' in n
    elif bname == 'EfficientNetB0':
        for n, p in model.named_parameters():
            p.requires_grad = 'head' in n
    elif bname == 'InceptionV3':
        for n, p in model.named_parameters():
            p.requires_grad = ('fc' in n or 'AuxLogits' in n)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f'  Phase 1 — head only,  {P1_EP} epochs,  LR=1e-3  '
          f'({trainable/1e3:.0f}K trainable)\n')
    print(f'  {"Ep":>5}  {"tr_loss":>9}  {"tr_acc":>8}  '
          f'{"vl_loss":>9}  {"vl_acc":>8}')
    print(f'  ' + '-'*48)

    opt1   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3, weight_decay=WD)
    sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt1, T_max=P1_EP, eta_min=1e-5)

    best_acc = 0.0

    for ep in range(1, P1_EP + 1):
        tl, ta = train_epoch(model, tr_ld, opt1, bname)
        vl, va = eval_epoch(model, vl_ld, bname)
        sched1.step()
        hist.append(('p1', tl, vl, ta, va))

        flag = ''
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), ckpt)
            flag = '  ✓ saved'

        print(f'  {ep:02d}/{P1_EP}  '
              f'{tl:>9.4f}  {ta:>8.4f}  '
              f'{vl:>9.4f}  {va:>8.4f}' + flag)

    # FIX 1: Always ensure checkpoint exists after Phase 1
    if not os.path.exists(ckpt):
        torch.save(model.state_dict(), ckpt)
        print('  (Phase 1: saved fallback checkpoint — val acc never improved)')

    print(f'\n  Phase 1 done — best val acc : {best_acc*100:.2f}%')

    # ── Phase 2: full network, 80 epochs ─────────────────────────
    P2_EP = 80

    print(f'\n  Phase 2 — full network,  {P2_EP} epochs,  LR=5e-5\n')

    # Load best Phase-1 checkpoint before unfreezing
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    for p in model.parameters():
        p.requires_grad = True

    trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable : {trainable2/1e6:.2f}M  (all layers unfrozen)\n')
    print(f'  {"Ep":>5}  {"tr_loss":>9}  {"tr_acc":>8}  '
          f'{"vl_loss":>9}  {"vl_acc":>8}')
    print(f'  ' + '-'*48)

    opt2   = torch.optim.Adam(model.parameters(), lr=5e-5, weight_decay=WD)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=P2_EP, eta_min=1e-7)

    best_acc = 0.0

    for ep in range(1, P2_EP + 1):
        tl, ta = train_epoch(model, tr_ld, opt2, bname)
        vl, va = eval_epoch(model, vl_ld, bname)
        sched2.step()
        hist.append(('p2', tl, vl, ta, va))

        flag = ''
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), ckpt)
            flag = '  ✓ saved'

        print(f'  {ep:02d}/{P2_EP}  '
              f'{tl:>9.4f}  {ta:>8.4f}  '
              f'{vl:>9.4f}  {va:>8.4f}' + flag)

    # FIX 1: Always save final-epoch model (fully trained 100-epoch model)
    # This guarantees Cell 7b always loads the complete trained model.
    torch.save(model.state_dict(), ckpt)
    print(f'\n  Phase 2 done — best val acc : {best_acc*100:.2f}%')
    print(f'  Total epochs completed      : {len(hist)}  (20 + 80)')
    print(f'  Checkpoint saved to         : {ckpt}')

    all_histories[bname] = hist

print('\n' + '='*62)
print('  ALL 4 CLASSIFIERS TRAINED — 100 epochs each')
print('='*62)



  TRAINING  ResNet18  →  295-class classification
  Training data : REAL images only
  Phase 1       : 20 epochs  (head only)
  Phase 2       : 80 epochs  (full network)
  Total         : EXACTLY 100 epochs — no early stop

  Params          : 11.4M
  Input size      : 224×224
  Random baseline : 0.34%

  Phase 1 — head only,  20 epochs,  LR=1e-3  (207K trainable)

     Ep    tr_loss    tr_acc    vl_loss    vl_acc
  ------------------------------------------------
  01/20     5.6975    0.0051     5.6268    0.0102  ✓ saved
  02/20     5.5583    0.0110     5.4015    0.0288  ✓ saved
  03/20     5.3583    0.0216     5.1923    0.0305  ✓ saved
  04/20     5.1814    0.0278     4.9166    0.0661  ✓ saved
  05/20     5.0283    0.0328     4.7278    0.0949  ✓ saved
  06/20     4.9018    0.0458     4.5357    0.1288  ✓ saved
  07/20     4.7789    0.0477     4.4241    0.1373  ✓ saved
  08/20     4.6666    0.0587     4.3056    0.1551  ✓ saved
  09/20     4.6277    0.0621     4.2017    0.1746  ✓ saved

In [9]:
# ── Checkpoint Verification ───────────────────────────────────────
# FIX: removed wrong SAVE_DIR = '/DATA/mtp/...' override that caused MISSING ✗
# SAVE_DIR is already correctly set from Cell 2/7.

print(f'Checking checkpoints in: {SAVE_DIR}\n')
all_ok = True
for bname in BACKBONES:
    p = os.path.join(SAVE_DIR, f'{bname}_cls.pt')
    exists = os.path.exists(p)
    status = 'EXISTS ✓' if exists else 'MISSING ✗'
    print(f'  {bname}_cls.pt : {status}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n  All checkpoints found — ready for spoof-aware fine-tuning ✓')
else:
    print('\n  WARNING: some checkpoints missing — re-run Cell 7 first!')


Checking checkpoints in: /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/outputs_295class

  ResNet18_cls.pt : EXISTS ✓
  ResNet50_cls.pt : EXISTS ✓
  EfficientNetB0_cls.pt : EXISTS ✓
  InceptionV3_cls.pt : EXISTS ✓

  All checkpoints found — ready for spoof-aware fine-tuning ✓


In [10]:
# ── Cell 7b: STEP 1b — Spoof-Aware Fine-Tuning ───────────────────
# FIX 1: SA_EPOCHS = 30 (was 20), early stopping REMOVED
# FIX 2: Triplet margin = 0.5 (was 0.3) → stronger separation
# FIX 3: SAVE_DIR re-confirmed here (guards against any override)
# FIX 4: EfficientNetB0 get_embedding uses correct forward path
# FIX 5: Final _sa.pt always saved at end (even if last epoch not best)

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Always re-confirm SAVE_DIR
SAVE_DIR = os.path.join(BASE, 'outputs_295class')


# ── Triplet Dataset ───────────────────────────────────────────────
class TripletSpoofDataset(Dataset):
    """
    Each item returns:
      anchor   : real image of subject i
      positive : different real image of subject i (or same if only 1)
      negative : spoof image of subject i
      label    : class index of subject i
    Only subjects with BOTH real AND spoof images are used.
    """
    def __init__(self, real_dir, attack_dir, subject_to_idx,
                 transform, img_size):
        self.transform = transform
        self.resize    = transforms.Resize((img_size, img_size))
        self.triplets  = []

        for subj, idx in subject_to_idx.items():
            rp = Path(real_dir)   / subj
            ap = Path(attack_dir) / subj
            if not rp.is_dir() or not ap.is_dir():
                continue

            real_imgs  = []
            spoof_imgs = []
            for ext in EXTS:
                real_imgs  += list(rp.glob(ext))
                spoof_imgs += list(ap.glob(ext))

            if len(real_imgs) == 0 or len(spoof_imgs) == 0:
                continue

            for r_path in real_imgs:
                pos_candidates = [p for p in real_imgs if p != r_path]
                pos_path = random.choice(pos_candidates) if pos_candidates else r_path
                neg_path = random.choice(spoof_imgs)
                self.triplets.append((str(r_path), str(pos_path), str(neg_path), idx))

    def __len__(self):
        return len(self.triplets)

    def _load(self, path):
        img = self.resize(Image.open(path).convert('RGB'))
        return self.transform(img)

    def __getitem__(self, idx):
        anc_p, pos_p, neg_p, label = self.triplets[idx]
        return (self._load(anc_p), self._load(pos_p),
                self._load(neg_p), torch.tensor(label, dtype=torch.long))


# ── Spoof-Aware Model Wrapper ─────────────────────────────────────
class SpoofAwareModel(nn.Module):
    """
    Wraps existing classifier, adds binary real/spoof head.
    FIX: get_embedding for EfficientNetB0 now uses global_pool
    so embedding space is consistent with what the head was trained on.
    """
    def __init__(self, backbone_name, base_model):
        super().__init__()
        self.backbone_name = backbone_name
        self.base          = base_model

        dim_map = {
            'ResNet18':       512,
            'ResNet50':       2048,
            'EfficientNetB0': 1280,
            'InceptionV3':    2048,
        }
        emb_dim = dim_map[backbone_name]

        self.spoof_head = nn.Sequential(
            nn.Linear(emb_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def get_embedding(self, x):
        """Extract raw feature vector before classification head."""
        b  = self.base
        bn = self.backbone_name

        if bn in ('ResNet18', 'ResNet50'):
            for name, layer in b.named_children():
                if name == 'fc':
                    break
                if name == 'avgpool':
                    x = layer(x)
                    x = torch.flatten(x, 1)
                else:
                    x = layer(x)
            return x

        if bn == 'EfficientNetB0':
            # FIX: was using forward_features only (pre-pool features)
            # Now uses global_pool → matches training embedding space
            feats = b.backbone.forward_features(x)   # (B, 1280, 7, 7)
            feats = b.backbone.global_pool(feats)     # (B, 1280)
            return feats

        if bn == 'InceptionV3':
            b.aux_logits = False
            x = b.Conv2d_1a_3x3(x);  x = b.Conv2d_2a_3x3(x)
            x = b.Conv2d_2b_3x3(x);  x = b.maxpool1(x)
            x = b.Conv2d_3b_1x1(x);  x = b.Conv2d_4a_3x3(x)
            x = b.maxpool2(x)
            x = b.Mixed_5b(x); x = b.Mixed_5c(x); x = b.Mixed_5d(x)
            x = b.Mixed_6a(x); x = b.Mixed_6b(x); x = b.Mixed_6c(x)
            x = b.Mixed_6d(x); x = b.Mixed_6e(x)
            x = b.Mixed_7a(x); x = b.Mixed_7b(x); x = b.Mixed_7c(x)
            x = b.avgpool(x);  x = b.dropout(x)
            x = torch.flatten(x, 1)
            return x

        raise ValueError(f'Unknown backbone: {bn}')

    def forward(self, anchor, positive, negative):
        anc_feat = self.get_embedding(anchor)
        pos_feat = self.get_embedding(positive)
        neg_feat = self.get_embedding(negative)

        anc_emb = F.normalize(anc_feat, p=2, dim=1)
        pos_emb = F.normalize(pos_feat, p=2, dim=1)
        neg_emb = F.normalize(neg_feat, p=2, dim=1)

        id_logits = self.base(anchor)
        if hasattr(id_logits, 'logits'):
            id_logits = id_logits.logits

        anc_spoof    = self.spoof_head(anc_feat)
        neg_spoof    = self.spoof_head(neg_feat)
        spoof_logits = torch.cat([anc_spoof, neg_spoof], dim=0)

        return anc_emb, pos_emb, neg_emb, id_logits, spoof_logits


# ── Combined Loss (FIX: triplet margin raised to 0.5) ────────────
# FIX: margin = 0.5 (was 0.3) → forces harder spoof/real separation
triplet_loss_fn = nn.TripletMarginWithDistanceLoss(
    distance_function=lambda a, b: 1.0 - F.cosine_similarity(a, b),
    margin=0.5
)
bce_loss_fn = nn.BCEWithLogitsLoss()
ce_loss_fn  = nn.CrossEntropyLoss()

W_CE      = 0.3
W_TRIPLET = 0.4
W_BINARY  = 0.3


def spoof_aware_loss(anc_emb, pos_emb, neg_emb,
                     id_logits, spoof_logits, labels, batch_size):
    loss_ce  = ce_loss_fn(id_logits, labels)
    loss_tri = triplet_loss_fn(anc_emb, pos_emb, neg_emb)
    binary_labels = torch.cat([
        torch.ones(batch_size,  1, device=DEVICE),
        torch.zeros(batch_size, 1, device=DEVICE)
    ], dim=0)
    loss_bin = bce_loss_fn(spoof_logits, binary_labels)
    total    = W_CE * loss_ce + W_TRIPLET * loss_tri + W_BINARY * loss_bin
    return total, loss_ce.item(), loss_tri.item(), loss_bin.item()


# ── Fine-Tuning Loop ──────────────────────────────────────────────
SA_EPOCHS = 30   # FIX: was 20 → more epochs for stronger convergence
SA_LR     = 2e-5

for bname in BACKBONES:
    ckpt_cls = os.path.join(SAVE_DIR, f'{bname}_cls.pt')
    ckpt_sa  = os.path.join(SAVE_DIR, f'{bname}_sa.pt')

    if not os.path.exists(ckpt_cls):
        print(f'  Skipping {bname} — _cls.pt not found. Run Cell 7 first.')
        continue

    print('\n' + '='*62)
    print(f'  SPOOF-AWARE FINE-TUNE  {bname}')
    print('='*62)

    train_tf, eval_tf, sz = get_transforms(bname)

    triplet_ds = TripletSpoofDataset(
        REAL_DIR, ATTACK_DIR, SUBJECT_TO_IDX, train_tf, sz)
    triplet_ld = DataLoader(
        triplet_ds, batch_size=BATCH_SIZE,
        shuffle=True, num_workers=0, pin_memory=True)

    print(f'  Triplet samples : {len(triplet_ds)}')
    print(f'  Batches/epoch   : {len(triplet_ld)}\n')

    base_model = build_classifier(bname, NUM_CLASSES).to(DEVICE)
    base_model.load_state_dict(torch.load(ckpt_cls, map_location=DEVICE))
    model = SpoofAwareModel(bname, base_model).to(DEVICE)

    # Phase A: warm up spoof_head (5 epochs, backbone frozen)
    for p in model.base.parameters():
        p.requires_grad = False
    for p in model.spoof_head.parameters():
        p.requires_grad = True

    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-3, weight_decay=WD)

    print('  Phase A — spoof head warmup (5 epochs, backbone frozen)')
    for ep in range(1, 6):
        model.train()
        tot = 0.0; nb = 0
        for anc, pos, neg, lbl in triplet_ld:
            anc = anc.to(DEVICE); pos = pos.to(DEVICE)
            neg = neg.to(DEVICE); lbl = lbl.to(DEVICE)
            opt.zero_grad()
            ae, pe, ne, id_l, sp_l = model(anc, pos, neg)
            loss, lce, ltri, lbin = spoof_aware_loss(
                ae, pe, ne, id_l, sp_l, lbl, len(lbl))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        print(f'  Ep{ep:02d} | loss={tot/nb:.4f}')

    # Phase B: full fine-tune, SA_EPOCHS epochs, NO early stopping
    for p in model.parameters():
        p.requires_grad = True

    opt   = torch.optim.Adam(model.parameters(), lr=SA_LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=SA_EPOCHS, eta_min=1e-6)

    print(f'\n  Phase B — full fine-tune ({SA_EPOCHS} epochs, LR={SA_LR}, NO early stop)')
    print(f'  {"Ep":>4}  {"total":>8}  {"ce":>8}  {"tri":>8}  {"bin":>8}')
    print('  ' + '-'*44)

    best_loss = float('inf')

    for ep in range(1, SA_EPOCHS + 1):
        model.train()
        tot = 0.0; lce_t = 0.0; ltri_t = 0.0; lbin_t = 0.0; nb = 0
        for anc, pos, neg, lbl in triplet_ld:
            anc = anc.to(DEVICE); pos = pos.to(DEVICE)
            neg = neg.to(DEVICE); lbl = lbl.to(DEVICE)
            opt.zero_grad()
            ae, pe, ne, id_l, sp_l = model(anc, pos, neg)
            loss, lce, ltri, lbin = spoof_aware_loss(
                ae, pe, ne, id_l, sp_l, lbl, len(lbl))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot   += loss.item()
            lce_t += lce; ltri_t += ltri; lbin_t += lbin
            nb    += 1
        sched.step()
        avg  = tot / nb
        flag = ''
        if avg < best_loss:
            best_loss = avg
            torch.save(model.base.state_dict(), ckpt_sa)
            flag = '  ✓ best'
        print(f'  {ep:02d}/{SA_EPOCHS}  {avg:>8.4f}  '
              f'{lce_t/nb:>8.4f}  {ltri_t/nb:>8.4f}  '
              f'{lbin_t/nb:>8.4f}' + flag)

    # FIX 5: Always save final model (even if last epoch wasn't best)
    torch.save(model.base.state_dict(), ckpt_sa)
    print(f'\n  Best loss     : {best_loss:.4f}')
    print(f'  SA checkpoint : {ckpt_sa}  ✓')

print('\n' + '='*62)
print('  SPOOF-AWARE FINE-TUNING DONE')
print('  Checkpoints saved: {bname}_sa.pt for each backbone')
print('='*62)



  SPOOF-AWARE FINE-TUNE  ResNet18
  Triplet samples : 5900
  Batches/epoch   : 185

  Phase A — spoof head warmup (5 epochs, backbone frozen)
  Ep01 | loss=0.4386
  Ep02 | loss=0.4133
  Ep03 | loss=0.4031
  Ep04 | loss=0.3921
  Ep05 | loss=0.3816

  Phase B — full fine-tune (30 epochs, LR=2e-05, NO early stop)
    Ep     total        ce       tri       bin
  --------------------------------------------
  01/30    0.3558    0.3508    0.3705    0.3413  ✓ best
  02/30    0.2968    0.3487    0.3245    0.2080  ✓ best
  03/30    0.2348    0.3403    0.2573    0.0994  ✓ best
  04/30    0.1876    0.3351    0.1750    0.0569  ✓ best
  05/30    0.1503    0.3141    0.1117    0.0381  ✓ best
  06/30    0.1270    0.3114    0.0578    0.0350  ✓ best
  07/30    0.1117    0.2926    0.0415    0.0245  ✓ best
  08/30    0.0930    0.2621    0.0251    0.0146  ✓ best
  09/30    0.0914    0.2636    0.0212    0.0129  ✓ best
  10/30    0.0882    0.2490    0.0162    0.0236  ✓ best
  11/30    0.0823    0.2358    0.

In [11]:
# ── Cell 8: STEP 2 — Extract Embeddings ──────────────────────────
# FIX: Now stores INDIVIDUAL image-level embeddings for RS pairs
# → instead of 295 RS pairs (subject mean vs subject mean),
#   we get ~20 × 295 ≈ 5,900 RS pairs (per-image real vs per-image spoof)
# → smoother, more Gaussian-like orange distribution in the plot
#
# For RR and RC we still use subject-mean embeddings (stable, low-noise).
# SAVE_DIR re-confirmed here too.

SAVE_DIR = os.path.join(BASE, 'outputs_295class')


@torch.no_grad()
def extract_all_embeddings(extractor, loader):
    """
    Extract per-image embeddings, grouped by subject and split.
    Returns dict: {subject: {'real': [emb1,...], 'attack': [emb1,...]}}
    Each emb is L2-normalised.
    """
    raw = defaultdict(lambda: {'real': [], 'attack': []})

    for imgs, subjects, splits in loader:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        feats = extractor(imgs)

        if isinstance(feats, tuple):
            feats = feats[0]
        if feats.dim() > 2:
            feats = feats.view(feats.size(0), -1)

        feats = F.normalize(feats, p=2, dim=1).cpu().numpy()

        for emb, subj, split in zip(feats, subjects, splits):
            raw[subj][split].append(emb)

    return raw


def compute_subject_mean_embeddings(raw_embs):
    """
    Average all per-image embeddings per subject → one mean vector.
    L2-normalise the mean again → unit-sphere gallery embedding.
    Used for RR (real-real) and RC (cross-spoof) comparisons.
    """
    real_emb  = {}
    spoof_emb = {}

    for subj, d in raw_embs.items():
        if d['real']:
            stacked  = np.stack(d['real'], axis=0)
            mean_emb = stacked.mean(axis=0)
            norm     = np.linalg.norm(mean_emb) + 1e-8
            real_emb[subj]  = mean_emb / norm

        if d['attack']:
            stacked  = np.stack(d['attack'], axis=0)
            mean_emb = stacked.mean(axis=0)
            norm     = np.linalg.norm(mean_emb) + 1e-8
            spoof_emb[subj] = mean_emb / norm

    return real_emb, spoof_emb


all_real_embs   = {}   # {backbone: {subject: mean_array}}
all_spoof_embs  = {}   # {backbone: {subject: mean_array}}
all_raw_embs    = {}   # {backbone: raw per-image dict} — for image-level RS pairs

for bname in BACKBONES:
    print(f'\n  Extracting embeddings: {bname}')

    ckpt_sa  = os.path.join(SAVE_DIR, f'{bname}_sa.pt')
    ckpt_cls = os.path.join(SAVE_DIR, f'{bname}_cls.pt')

    if os.path.exists(ckpt_sa):
        ckpt = ckpt_sa
        print(f'  Using _sa.pt (spoof-aware) ✓')
    elif os.path.exists(ckpt_cls):
        ckpt = ckpt_cls
        print(f'  Using _cls.pt (fallback) ✓')
    else:
        print(f'  No checkpoint found — skipping ✗')
        continue

    _, eval_tf, sz = get_transforms(bname)
    emb_loader     = DataLoader(
        SubjectImageDataset(all_emb_samples, eval_tf, sz),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    extractor = build_extractor(bname, ckpt)
    raw_embs  = extract_all_embeddings(extractor, emb_loader)

    real_emb, spoof_emb = compute_subject_mean_embeddings(raw_embs)

    all_real_embs[bname]  = real_emb
    all_spoof_embs[bname] = spoof_emb
    all_raw_embs[bname]   = raw_embs

    n_imgs = {s: len(raw_embs[s]['real']) for s in raw_embs if raw_embs[s]['real']}
    avg_i  = np.mean(list(n_imgs.values())) if n_imgs else 0

    # Count image-level RS pairs (FIX: much more than 295)
    rs_img_pairs = sum(
        len(raw_embs[s]['real']) * len(raw_embs[s]['attack'])
        for s in raw_embs
        if raw_embs[s]['real'] and raw_embs[s]['attack']
    )

    print(f'  Real  subject embeddings     : {len(real_emb)}')
    print(f'  Spoof subject embeddings     : {len(spoof_emb)}')
    print(f'  Avg images per subject       : {avg_i:.1f}')
    print(f'  Image-level RS pairs (new)   : {rs_img_pairs:,}')

print('\n  Embeddings extracted.')



  Extracting embeddings: ResNet18
  Using _sa.pt (spoof-aware) ✓
  Real  subject embeddings     : 295
  Spoof subject embeddings     : 295
  Avg images per subject       : 20.0
  Image-level RS pairs (new)   : 118,000

  Extracting embeddings: ResNet50
  Using _sa.pt (spoof-aware) ✓
  Real  subject embeddings     : 295
  Spoof subject embeddings     : 295
  Avg images per subject       : 20.0
  Image-level RS pairs (new)   : 118,000

  Extracting embeddings: EfficientNetB0
  Using _sa.pt (spoof-aware) ✓
  Real  subject embeddings     : 295
  Spoof subject embeddings     : 295
  Avg images per subject       : 20.0
  Image-level RS pairs (new)   : 118,000

  Extracting embeddings: InceptionV3
  Using _sa.pt (spoof-aware) ✓
  Real  subject embeddings     : 295
  Spoof subject embeddings     : 295
  Avg images per subject       : 20.0
  Image-level RS pairs (new)   : 118,000

  Embeddings extracted.


In [12]:
# ── Cell 9: STEP 3 — Pairwise Dissimilarity ──────────────────────
# FIX: Version B now uses IMAGE-LEVEL RS pairs
#   → ~5,900 pairs instead of 295
#   → orange distribution has enough samples to be smooth & Gaussian
#
# Version A (RR) and C (RC) still use subject-mean embeddings.

def dissim(a, b):
    """Cosine dissimilarity between two L2-normalised vectors."""
    return float(1.0 - np.dot(a, b))


def compute_all_dissimilarities(real_emb, spoof_emb, raw_embs):
    """
    Ver A: Real-mean vs Real-mean (all pairs i < j)
    Ver B: EVERY real image vs EVERY spoof image of SAME subject (image-level)
    Ver C: Real-mean vs Spoof-mean DIFFERENT subject (cross)
    """
    real_subjects_list  = sorted(real_emb.keys())
    spoof_subjects_list = sorted(spoof_emb.keys())

    # ── VERSION A: Real-mean vs Real-mean (all pairs i < j) ───────
    rr_scores = []
    rr_pairs  = []
    for i, si in enumerate(real_subjects_list):
        for j, sj in enumerate(real_subjects_list):
            if j <= i: continue
            d = dissim(real_emb[si], real_emb[sj])
            rr_scores.append(d)
            rr_pairs.append((si, sj, d))

    # ── VERSION B: Image-level Real vs Spoof SAME subject (FIX) ───
    # Each real image paired with EACH spoof image of SAME subject
    rs_scores = []
    rs_pairs  = []
    matched   = sorted(set(real_subjects_list) & set(spoof_subjects_list))
    for subj in matched:
        real_imgs  = raw_embs[subj]['real']    # list of np arrays
        spoof_imgs = raw_embs[subj]['attack']  # list of np arrays
        if not real_imgs or not spoof_imgs:
            continue
        for re in real_imgs:
            for se in spoof_imgs:
                # Both are already L2-normalised
                d = float(1.0 - np.dot(re, se))
                rs_scores.append(d)
                rs_pairs.append((subj, d))

    # ── VERSION C: Real-mean vs Spoof-mean DIFFERENT subject ───────
    rc_scores = []
    rc_pairs  = []
    for si in real_subjects_list:
        for sj in spoof_subjects_list:
            if si == sj: continue
            d = dissim(real_emb[si], spoof_emb[sj])
            rc_scores.append(d)
            rc_pairs.append((si, sj, d))

    return (np.array(rr_scores), np.array(rs_scores), np.array(rc_scores),
            rr_pairs, rs_pairs, rc_pairs)


all_dissim = {}

for bname in BACKBONES:
    if bname not in all_real_embs:
        continue

    print(f'\n  Computing dissimilarity: {bname}')
    real_emb  = all_real_embs[bname]
    spoof_emb = all_spoof_embs[bname]
    raw_embs  = all_raw_embs[bname]

    rr, rs, rc, rr_p, rs_p, rc_p = compute_all_dissimilarities(
        real_emb, spoof_emb, raw_embs)

    if len(rs) > 0:
        mu_rr = float(np.mean(rr))
        mu_rs = float(np.mean(rs))
        sd_rr = float(np.std(rr))
        sd_rs = float(np.std(rs))

        # d' = (RR_mean - RS_mean) / pooled_std → POSITIVE = correct
        d_prime = (mu_rr - mu_rs) / (
            np.sqrt(0.5 * (sd_rr**2 + sd_rs**2)) + 1e-8)

        gap    = mu_rr - mu_rs
        rr_q25 = float(np.percentile(rr, 25))
        rs_arr = np.array(rs)
        vuln_high = float(np.mean(rs_arr < 0.10))
        vuln_med  = float(np.mean(rs_arr < 0.20))
        vuln_rel  = float(np.mean(rs_arr < mu_rr))
    else:
        d_prime = gap = rr_q25 = vuln_high = vuln_med = vuln_rel = float('nan')

    all_dissim[bname] = {
        'rr': rr, 'rs': rs, 'rc': rc,
        'rr_pairs': rr_p, 'rs_pairs': rs_p, 'rc_pairs': rc_p,
        'd_prime':   d_prime,
        'gap':       gap,
        'rr_q25':    rr_q25,
        'vuln_high': vuln_high,
        'vuln_med':  vuln_med,
        'vuln_rel':  vuln_rel,
        'rr_mean':   float(np.mean(rr)),
        'rs_mean':   float(np.mean(rs)) if len(rs) > 0 else float('nan'),
        'rc_mean':   float(np.mean(rc)) if len(rc) > 0 else float('nan'),
        'rr_std':    float(np.std(rr)),
        'rs_std':    float(np.std(rs))  if len(rs) > 0 else float('nan'),
    }

    print(f'  Ver A — Real-Real (subject mean) : {len(rr):6,} pairs  '
          f'mean={np.mean(rr):.4f}  std={np.std(rr):.4f}')
    print(f'  Ver B — Real-Spoof SAME (img-lvl): {len(rs):6,} pairs  '
          f'mean={np.mean(rs):.4f}  std={np.std(rs):.4f}')
    print(f'  Ver C — Cross-Spoof (subj mean)  : {len(rc):6,} pairs  '
          f'mean={np.mean(rc):.4f}  std={np.std(rc):.4f}')
    print(f"  d' (positive = good)             : {d_prime:.4f}")
    print(f"  Gap (RR - RS)                    : {gap:.4f}  "
          f"{'✓ GOOD' if gap > 0.15 else '⚠ WEAK'}")
    print(f"  Vuln HIGH (dissim<0.10)          : {vuln_high*100:.1f}%")
    print(f"  Vuln MED  (dissim<0.20)          : {vuln_med*100:.1f}%")
    print(f"  Vuln REL  (below RR mean)        : {vuln_rel*100:.1f}%")

    if rs_p:
        # Aggregate to subject-level for vulnerability report
        subj_rs = defaultdict(list)
        for subj, d in rs_p:
            subj_rs[subj].append(d)
        subj_mean_rs = {s: np.mean(v) for s, v in subj_rs.items()}
        sorted_rs    = sorted(subj_mean_rs.items(), key=lambda x: x[1])

        print(f'\n  Top-10 most VULNERABLE subjects (lowest avg real-spoof dissimilarity):')
        print(f'  {"Subject":<20}  {"AvgDissim":>10}  Interpretation')
        print(f'  ' + '-'*55)
        for subj, d in sorted_rs[:10]:
            vuln = ('HIGH RISK' if d < 0.10 else
                    'MEDIUM'    if d < 0.20 else
                    'LOW RISK')
            print(f'  {subj:<20}  {d:>10.4f}  {vuln}')

# ── Summary Table ─────────────────────────────────────────────────
print('\n' + '='*78)
print('  COMPARISON TABLE — Dissimilarity (image-level RS)')
print('='*78)
print(f'  {"Backbone":<18}  {"RR mean":>8}  {"RS mean":>8}  '
      f'{"Gap":>8}  {"d-prime":>8}  {"#RR":>7}  {"#RS":>8}')
print('  ' + '-'*78)
for bname in BACKBONES:
    if bname not in all_dissim: continue
    r = all_dissim[bname]
    print(f'  {bname:<18}  {r["rr_mean"]:>8.4f}  {r["rs_mean"]:>8.4f}  '
          f'{r["gap"]:>8.4f}  {r["d_prime"]:>8.4f}  '
          f'{len(r["rr"]):>7,}  {len(r["rs"]):>8,}')
print('  ' + '-'*78)
print('  #RS is now image-level pairs (much more than 295 → smoother orange dist)')
print('  d-prime > 0 means model correctly separates real from spoof')



  Computing dissimilarity: ResNet18
  Ver A — Real-Real (subject mean) : 43,365 pairs  mean=0.4397  std=0.1028
  Ver B — Real-Spoof SAME (img-lvl): 118,000 pairs  mean=0.3607  std=0.2142
  Ver C — Cross-Spoof (subj mean)  : 86,730 pairs  mean=0.4546  std=0.1284
  d' (positive = good)             : 0.4702
  Gap (RR - RS)                    : 0.0790  ⚠ WEAK
  Vuln HIGH (dissim<0.10)          : 7.7%
  Vuln MED  (dissim<0.20)          : 28.8%
  Vuln REL  (below RR mean)        : 67.1%

  Top-10 most VULNERABLE subjects (lowest avg real-spoof dissimilarity):
  Subject                AvgDissim  Interpretation
  -------------------------------------------------------
  450                       0.0765  HIGH RISK
  434                       0.0783  HIGH RISK
  398                       0.0926  HIGH RISK
  433                       0.0954  HIGH RISK
  396                       0.0964  HIGH RISK
  442                       0.1018  MEDIUM
  469                       0.1024  MEDIUM
  503         

In [13]:
# ── Cell 10: Visualisations ───────────────────────────────────────
# FIX: Added KDE (kernel density estimate) curves over histograms
# → orange distribution now shows smooth curve even with many RS pairs
# → easy to visually check Gaussian shape

print('\nGenerating plots...')
SAVE_DIR = os.path.join(BASE, 'outputs_295class')
avail    = [b for b in BACKBONES if b in all_dissim]


# ── Figure 1: Dissimilarity distributions + KDE curves ────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Subject-wise Dissimilarity Distributions\n'
    'Blue = Real-Real (Ver A)  |  '
    'Orange = Real-Spoof same (Ver B, image-level)  |  '
    'Green = Real-Spoof cross (Ver C)',
    fontsize=12, fontweight='bold')

for ax, bname in zip(axes.flat, avail):
    r   = all_dissim[bname]
    rr  = r['rr']
    rs  = r['rs']
    rc  = r['rc']

    # Histogram (density)
    ax.hist(rr, bins=60, alpha=0.4, color='steelblue', density=True,
            label=f'Real-Real  (μ={r["rr_mean"]:.3f})')
    if len(rs) > 1:
        ax.hist(rs, bins=60, alpha=0.4, color='coral', density=True,
                label=f'Real-Spoof same (μ={r["rs_mean"]:.3f})')
    if len(rc) > 1:
        ax.hist(rc, bins=60, alpha=0.3, color='green', density=True,
                label=f'Real-Spoof cross (μ={r["rc_mean"]:.3f})')

    # KDE curves — smoother visual
    x_grid = np.linspace(0.0, 1.4, 400)
    try:
        kde_rr = gaussian_kde(rr, bw_method='scott')
        ax.plot(x_grid, kde_rr(x_grid), color='steelblue', lw=2.0)
    except Exception: pass
    try:
        if len(rs) > 1:
            kde_rs = gaussian_kde(rs, bw_method='scott')
            ax.plot(x_grid, kde_rs(x_grid), color='coral', lw=2.5,
                    label='_nolegend_')
    except Exception: pass
    try:
        if len(rc) > 1:
            kde_rc = gaussian_kde(rc, bw_method='scott')
            ax.plot(x_grid, kde_rc(x_grid), color='green', lw=2.0,
                    label='_nolegend_')
    except Exception: pass

    ax.set_title(
        f'{bname}\nd\'={r["d_prime"]:.3f}  '
        f'(RS pairs: {len(rs):,})',
        fontweight='bold', fontsize=10)
    ax.set_xlabel('Dissimilarity  (1 − cosine_similarity)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_xlim([0, 1.3])

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dissim_histograms.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('  Saved: dissim_histograms.png')


# ── Figure 2: Per-subject vulnerability bar chart ─────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle(
    'Per-Subject Avg Real-vs-Spoof Dissimilarity (Version B)\n'
    'Red = HIGH RISK (< 0.2)  |  Orange = MEDIUM (0.2–0.4)  |  Green = LOW RISK',
    fontsize=12, fontweight='bold')

for ax, bname in zip(axes.flat, avail):
    r = all_dissim[bname]
    if len(r['rs_pairs']) == 0:
        ax.set_title(f'{bname} — no matched subjects'); continue

    # Aggregate to subject-level mean for this bar chart
    subj_rs = defaultdict(list)
    for subj, d in r['rs_pairs']:
        subj_rs[subj].append(d)
    subj_mean = {s: np.mean(v) for s, v in subj_rs.items()}
    rs_sorted = sorted(subj_mean.items(), key=lambda x: x[1])

    subjects = [p[0] for p in rs_sorted]
    scores   = [p[1] for p in rs_sorted]
    bar_colors = ['#E24B4A' if s < 0.2 else
                  '#EF9F27' if s < 0.4 else '#639922'
                  for s in scores]
    x_pos = np.arange(len(subjects))
    ax.bar(x_pos, scores, color=bar_colors, alpha=0.85,
           edgecolor='white', linewidth=0.5)
    ax.axhline(0.2, color='red',    ls='--', lw=1.2, label='High-risk (0.2)')
    ax.axhline(0.4, color='orange', ls='--', lw=1.2, label='Medium (0.4)')
    ax.set_title(f'{bname}', fontweight='bold', fontsize=10)
    ax.set_xlabel('Subject (sorted by avg dissimilarity)')
    ax.set_ylabel('Avg Dissimilarity')
    ax.set_xticks([])
    ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)
    n_high = sum(1 for s in scores if s < 0.2)
    n_med  = sum(1 for s in scores if 0.2 <= s < 0.4)
    n_low  = sum(1 for s in scores if s >= 0.4)
    ax.text(0.02, 0.97,
            f'High: {n_high}  Medium: {n_med}  Low: {n_low}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'per_subject_vulnerability.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('  Saved: per_subject_vulnerability.png')


# ── Figure 3: d-prime bar chart ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bnames = [b for b in avail if b in all_dissim]
dps    = [all_dissim[b]['d_prime'] for b in bnames]
bars   = ax.bar(bnames, dps,
                color=[COLORS[b] for b in bnames],
                alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, dp in zip(bars, dps):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.02 if dp >= 0 else -0.08),
            f'{dp:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title("d' — Real-Real vs Real-Spoof separation\n"
             "positive = model correctly separates identities",
             fontweight='bold')
ax.set_ylabel("d-prime"); ax.grid(axis='y', alpha=0.3)
ax.set_xticklabels(bnames, rotation=10, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'dprime_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('  Saved: dprime_comparison.png')


# ── Figure 4: Real-Real heatmap (best backbone) ───────────────────
best_b    = bnames[dps.index(max(dps))]
real_emb  = all_real_embs[best_b]
subj_list = sorted(real_emb.keys())
n         = len(subj_list)
D         = np.zeros((n, n))
for i, si in enumerate(subj_list):
    for j, sj in enumerate(subj_list):
        D[i, j] = 0.0 if i == j else dissim(real_emb[si], real_emb[sj])

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(D, cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Dissimilarity')
ax.set_title(f'Real-Real Dissimilarity Matrix — {best_b}\n'
             f'({n} subjects × {n} subjects)',
             fontweight='bold')
ax.set_xlabel('Subject index'); ax.set_ylabel('Subject index')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'real_real_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print(f'  Saved: real_real_heatmap.png  ({best_b})')


# ── Figure 5: Training curves ─────────────────────────────────────
if all_histories:
    fig, axes = plt.subplots(2, 4, figsize=(22, 9))
    fig.suptitle('295-Class Classifier Training Curves (100 epochs each)',
                 fontsize=13, fontweight='bold')
    for col, bname in enumerate(BACKBONES):
        if bname not in all_histories: continue
        hist = all_histories[bname]
        trl  = [h[1] for h in hist]; vll = [h[2] for h in hist]
        tra  = [h[3] for h in hist]; vla = [h[4] for h in hist]
        eps  = range(1, len(hist) + 1)
        p1e  = sum(1 for h in hist if h[0] == 'p1')
        c    = COLORS[bname]

        ax = axes[0][col]
        ax.plot(eps, trl, color=c, lw=2, label='Train')
        ax.plot(eps, vll, color=c, lw=2, ls='--', label='Val')
        ax.axvline(p1e + 0.5, color='gray', ls=':', alpha=0.6, label='P1→P2')
        ax.set_title(f'{bname}\nCE Loss', fontsize=9, fontweight='bold')
        ax.legend(fontsize=7); ax.grid(alpha=0.3); ax.set_xlabel('Epoch')

        ax = axes[1][col]
        ax.plot(eps, tra, color=c, lw=2, label='Train')
        ax.plot(eps, vla, color=c, lw=2, ls='--', label='Val')
        ax.axvline(p1e + 0.5, color='gray', ls=':', alpha=0.6)
        ax.set_ylim([0, 1.02])
        ax.set_title(f'{bname}\nAccuracy', fontsize=9, fontweight='bold')
        ax.legend(fontsize=7); ax.grid(alpha=0.3); ax.set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print('  Saved: training_curves.png')


# ── Figure 6: RS distribution per backbone (zoomed in) ────────────
fig, axes = plt.subplots(1, len(avail), figsize=(5*len(avail), 4))
if len(avail) == 1: axes = [axes]
fig.suptitle('Real-Spoof Same-Subject Distribution (Ver B, image-level)\n'
             'Orange = histogram  |  Red curve = KDE  |  '
             'Dashed = mean',
             fontsize=12, fontweight='bold')

for ax, bname in zip(axes, avail):
    rs = all_dissim[bname]['rs']
    if len(rs) < 2:
        ax.set_title(f'{bname}\n(no RS pairs)'); continue
    ax.hist(rs, bins=60, density=True, color='coral', alpha=0.6,
            edgecolor='white', linewidth=0.3)
    x_g = np.linspace(max(0, rs.min()-0.05), min(1.5, rs.max()+0.05), 400)
    try:
        kde = gaussian_kde(rs, bw_method='scott')
        ax.plot(x_g, kde(x_g), color='red', lw=2.5)
    except Exception: pass
    ax.axvline(rs.mean(), color='darkred', ls='--', lw=1.5,
               label=f'mean={rs.mean():.3f}')
    ax.axvline(rs.mean() + rs.std(), color='darkred', ls=':', lw=1,
               label=f'±1σ={rs.std():.3f}')
    ax.axvline(rs.mean() - rs.std(), color='darkred', ls=':', lw=1)
    ax.set_title(f'{bname}\nn={len(rs):,} pairs', fontweight='bold')
    ax.set_xlabel('Dissimilarity (1 - cosine)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'rs_distribution_zoom.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('  Saved: rs_distribution_zoom.png')

print(f'\nAll outputs → {os.path.abspath(SAVE_DIR)}')
print('  training_curves.png')
print('  dissim_histograms.png')
print('  rs_distribution_zoom.png')
print('  per_subject_vulnerability.png')
print('  dprime_comparison.png')
print('  real_real_heatmap.png')



Generating plots...
  Saved: dissim_histograms.png
  Saved: per_subject_vulnerability.png
  Saved: dprime_comparison.png
  Saved: real_real_heatmap.png  (ResNet50)
  Saved: training_curves.png
  Saved: rs_distribution_zoom.png

All outputs → /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/outputs_295class
  training_curves.png
  dissim_histograms.png
  rs_distribution_zoom.png
  per_subject_vulnerability.png
  dprime_comparison.png
  real_real_heatmap.png


In [14]:
# ── Cell 11: Save Text Report ─────────────────────────────────────
SAVE_DIR = os.path.join(BASE, 'outputs_295class')
out_txt  = os.path.join(SAVE_DIR, 'dissimilarity_report.txt')

with open(out_txt, 'w') as f:
    f.write('295-Class Classification → Subject-wise Embedding → Pairwise Dissimilarity\n')
    f.write(f'Dataset : {os.path.abspath(BASE)}\n')
    f.write('='*72 + '\n\n')

    f.write('KEY FIXES APPLIED:\n')
    f.write('  1. Cell 11 SAVE_DIR override bug removed\n')
    f.write('  2. Final-epoch checkpoint always saved (no silent miss)\n')
    f.write('  3. Cell 7b SAVE_DIR re-confirmed → SA always runs\n')
    f.write('  4. SA_EPOCHS = 30, no early stopping\n')
    f.write('  5. Triplet margin = 0.5 (stronger spoof separation)\n')
    f.write('  6. EfficientNetB0 extractor uses global_pool (correct space)\n')
    f.write('  7. RS pairs: image-level (~5900) not subject-level (295)\n')
    f.write('  8. KDE overlay on histograms for smooth visual\n\n')

    f.write(f'{"Backbone":<18}  {"RR mean":>8}  {"RS mean":>8}  '
            f'{"RC mean":>8}  {"d-prime":>8}  {"#RR":>7}  {"#RS":>8}\n')
    f.write('-'*78 + '\n')
    for bname in BACKBONES:
        if bname not in all_dissim: continue
        r = all_dissim[bname]
        f.write(f'{bname:<18}  {r["rr_mean"]:>8.4f}  {r["rs_mean"]:>8.4f}  '
                f'{r["rc_mean"]:>8.4f}  {r["d_prime"]:>8.4f}  '
                f'{len(r["rr"]):>7,}  {len(r["rs"]):>8,}\n')
    f.write('-'*78 + '\n\n')

    for bname in BACKBONES:
        if bname not in all_dissim: continue
        r = all_dissim[bname]
        f.write(f'{bname}:\n')
        f.write(f'  RR (subject-mean pairs) : n={len(r["rr"]):,}  '
                f'mean={r["rr_mean"]:.4f}  std={r["rr_std"]:.4f}\n')
        f.write(f'  RS (image-level pairs)  : n={len(r["rs"]):,}  '
                f'mean={r["rs_mean"]:.4f}  std={r["rs_std"]:.4f}\n')
        f.write(f'  RC (cross-subj pairs)   : n={len(r["rc"]):,}  '
                f'mean={r["rc_mean"]:.4f}\n')
        f.write(f"  d'                      : {r['d_prime']:.4f}\n")
        f.write(f"  Gap (RR-RS)             : {r['gap']:.4f}\n\n")

print(f'Report saved → {out_txt}')
print('\n' + '='*62)
print('  ALL DONE')
print('  Step 1 : 295-class training  (100 epochs each) ✓')
print('  Step 1b: Spoof-aware fine-tune (30 epochs)    ✓')
print('  Step 2 : Image-level embedding extraction     ✓')
print('  Step 3 : Pairwise dissimilarity               ✓')
print('  Plots  : dissim_histograms + KDE overlay      ✓')
print('='*62)


Report saved → /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean/outputs_295class/dissimilarity_report.txt

  ALL DONE
  Step 1 : 295-class training  (100 epochs each) ✓
  Step 1b: Spoof-aware fine-tune (30 epochs)    ✓
  Step 2 : Image-level embedding extraction     ✓
  Step 3 : Pairwise dissimilarity               ✓
  Plots  : dissim_histograms + KDE overlay      ✓
